# 핸즈온 03 — VDS 시계열 분석 & Long Context & Caching

**소요 시간**: 70~80분
**학습 목표**:
1. **1M 컨텍스트 윈도우**를 활용해 대용량 시계열 데이터를 RAG 없이 통째로 분석한다
2. **Context Caching**으로 반복 질의 비용을 75% 절감한다
3. ITS 영역에서 가장 흔한 워크로드인 VDS(차량검지기) 시계열 데이터를 자연어로 질의한다

> **데이터 출처**: 본 핸즈온은 데이터 가용성을 위해 **합성 VDS 데이터**를 생성합니다. 실제 운영에서는 [공공데이터포털 - 도로공사 교통량](https://www.data.go.kr/data/15003078/openapi.do) 또는 [서울 열린데이터광장 - 도로별 교통량](https://data.seoul.go.kr/) API에서 동일한 형식의 데이터를 받을 수 있습니다.

## 3-1. 환경 셋업

In [ ]:
!pip install -q -U google-genai pandas numpy

In [ ]:
import os, json, time
from datetime import datetime, timedelta
from io import StringIO
import pandas as pd
import numpy as np
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("✅ Client ready")

## 3-2. VDS 시계열 데이터 생성 (1주일치, 5분 간격)

실제 도로공사 VDS 데이터 형식을 모방한 합성 데이터입니다.
- 5개 검지기 × 7일 × 288 (5분 간격) = 약 1만 행
- 컬럼: `timestamp`, `vds_id`, `volume`(교통량), `speed`(평균속도), `occupancy`(점유율)

이상 패턴을 일부러 심어둡니다 — 청중이 자연어 질의로 찾아낼 수 있는지 봅니다.

In [ ]:
np.random.seed(42)

VDS_INFO = {
    "VDS-0101": {"name": "경부선 서울TG 상행", "lanes": 4},
    "VDS-0102": {"name": "경부선 서울TG 하행", "lanes": 4},
    "VDS-0203": {"name": "영동선 안산JC", "lanes": 3},
    "VDS-0301": {"name": "서해안선 송도IC", "lanes": 3},
    "VDS-0405": {"name": "중부선 호법JC", "lanes": 4},
}

def synthesize_vds(vds_id, start_date, days=7):
    """5분 간격 1주일치 데이터 생성."""
    rows = []
    timestamps = pd.date_range(start_date, periods=days*288, freq="5min")
    for ts in timestamps:
        h = ts.hour + ts.minute/60
        # 일반적인 일중 패턴: 출근(08), 퇴근(18) 피크
        base_volume = 100 + 80 * np.exp(-((h-8)**2)/4) + 90 * np.exp(-((h-18)**2)/5)
        # 주말 효과
        if ts.weekday() >= 5:
            base_volume *= 0.6
        # 노이즈
        volume = max(0, int(base_volume + np.random.normal(0, 15)))
        # 속도는 교통량과 음의 상관
        speed = max(20, 100 - volume * 0.3 + np.random.normal(0, 5))
        # 점유율
        occupancy = min(95, volume * 0.4 + np.random.normal(0, 3))
        rows.append({
            "timestamp": ts.strftime("%Y-%m-%d %H:%M"),
            "vds_id": vds_id,
            "volume": volume,
            "speed": round(speed, 1),
            "occupancy": round(occupancy, 1),
        })
    return rows

# 1주일치 (2026-04-20 월 ~ 2026-04-26 일)
START = datetime(2026, 4, 20)
all_rows = []
for vds_id in VDS_INFO:
    all_rows.extend(synthesize_vds(vds_id, START, days=7))

df = pd.DataFrame(all_rows)

# 이상 패턴 주입: 4월 23일(목) 14:00~16:00 서해안선 사고 발생
mask = (df["vds_id"] == "VDS-0301") & (df["timestamp"] >= "2026-04-23 14:00") & (df["timestamp"] <= "2026-04-23 16:00")
df.loc[mask, "volume"] = (df.loc[mask, "volume"] * 0.2).astype(int)
df.loc[mask, "speed"] = df.loc[mask, "speed"] * 0.3
df.loc[mask, "occupancy"] = df.loc[mask, "occupancy"] * 2.5

# 이상 패턴 주입: 4월 25일(토) 영동선 야간 정체 (교통량은 정상이나 속도만 급락)
mask2 = (df["vds_id"] == "VDS-0203") & (df["timestamp"] >= "2026-04-25 22:00") & (df["timestamp"] <= "2026-04-26 02:00")
df.loc[mask2, "speed"] = df.loc[mask2, "speed"] * 0.4

print(f"총 행 수: {len(df):,}")
print(f"기간: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"검지기: {df['vds_id'].nunique()}개")
df.head()

## 3-3. Long Context로 통째로 던지기

10,080행 CSV를 그대로 컨텍스트에 넣으면 토큰이 얼마나 쓰일까요?

In [ ]:
# CSV 문자열로 변환
csv_text = df.to_csv(index=False)
print(f"CSV 크기: {len(csv_text):,} chars")

# 토큰 수 추정 (Gemini는 보통 4글자=1토큰)
print(f"추정 토큰: ~{len(csv_text) // 4:,}")

# 정확한 토큰 수 측정
token_count = client.models.count_tokens(
    model="gemini-3-flash-preview",
    contents=csv_text,
)
print(f"실측 토큰: {token_count.total_tokens:,}")

### 검지기 정보 (메타데이터) 포함

In [ ]:
METADATA = "## 검지기 정보\n"
for vid, info in VDS_INFO.items():
    METADATA += f"- {vid}: {info['name']} ({info['lanes']}차로)\n"

print(METADATA)

## 3-4. 자연어 질의 1 — 이상 패턴 자동 검출

In [ ]:
SYSTEM_INSTRUCTION = """당신은 한국도로공사 교통관제센터의 데이터 분석 엔지니어입니다.
VDS(차량검지기)에서 수집한 5분 간격 시계열 데이터를 분석합니다.
- volume: 5분간 통과 차량 대수
- speed: 평균 속도 (km/h)
- occupancy: 점유율 (%)
정확하고 간결하게, 필요시 구체적 시간/검지기를 명시하세요."""

QUERY = """이 1주일치 VDS 데이터에서 평소와 다른 이상 패턴을 모두 찾아주세요.
각 이상 패턴에 대해:
1. 어느 검지기에서 발생했는지
2. 언제 발생했는지 (날짜와 시간)
3. 어떤 패턴인지 (수치 근거 포함)
4. 추정 원인
을 정리해주세요."""

t0 = time.time()
resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=f"{METADATA}\n\n## 데이터\n\n{csv_text}\n\n## 질문\n{QUERY}",
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
    ),
)
elapsed = time.time() - t0

print(f"⏱  {elapsed:.1f}s | tokens in/out: {resp.usage_metadata.prompt_token_count:,}/{resp.usage_metadata.candidates_token_count:,}\n")
print(resp.text)

> **체크 포인트**
>
> 1. 4월 23일 서해안선 사고 패턴을 잡았는가?
> 2. 4월 25일 영동선 야간 속도 급락을 잡았는가? (교통량은 정상이라 더 어려움)
> 3. 잘못 짚은 이상 패턴(false positive)이 있는가?

## 3-5. 자연어 질의 2 — 운영 보고서 자동 생성

In [ ]:
REPORT_QUERY = """이 1주일 데이터를 바탕으로 주간 교통 운영 보고서를 작성해주세요.

다음 구성으로 작성:
1. 한 주 요약 (3~4줄)
2. 검지기별 평균 통행량 순위
3. 출퇴근 피크 시간대 분석 (전 검지기 평균)
4. 주중 vs 주말 교통량 비교
5. 발견된 이상 패턴
6. 다음 주 운영에 반영할 권고사항 3가지

마크다운 형식으로 작성하세요."""

resp2 = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=f"{METADATA}\n\n## 데이터\n\n{csv_text}\n\n## 작성 요청\n{REPORT_QUERY}",
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
    ),
)
print(resp2.text)

## 3-6. Context Caching — 같은 데이터에 여러 질의

위에서 우리는 같은 1만 행 데이터를 두 번이나 입력 토큰으로 보냈습니다. 이건 **비용 누수**입니다.

Explicit Caching을 사용하면 데이터를 한 번만 업로드하고 여러 질의에서 재사용할 수 있습니다.

> **제약사항**: Explicit Caching은 **최소 32,768 토큰** 이상의 컨텍스트만 캐시 가능합니다. 우리 데이터가 이 기준을 충족하는지 먼저 확인합니다.

In [ ]:
print(f"VDS 데이터 토큰: {token_count.total_tokens:,}")
print(f"최소 캐시 토큰:  32,768")
print(f"캐시 가능: {'✅' if token_count.total_tokens >= 32768 else '❌ 데이터 부족'}")

if token_count.total_tokens < 32768:
    print("\n  → 7일치로는 부족할 수 있으니 14일치로 늘려서 캐시 시연합니다.")

### 14일치로 데이터 확장 (필요시)

In [ ]:
# 14일치로 확장하여 32K 토큰 이상 확보
all_rows_14d = []
for vds_id in VDS_INFO:
    all_rows_14d.extend(synthesize_vds(vds_id, START, days=14))

df14 = pd.DataFrame(all_rows_14d)
csv_text_14d = df14.to_csv(index=False)
token_14d = client.models.count_tokens(
    model="gemini-3-flash-preview", contents=csv_text_14d
).total_tokens

print(f"14일치 토큰: {token_14d:,}")
print(f"캐시 가능: {'✅' if token_14d >= 32768 else '❌'}")

### 캐시 생성

In [ ]:
# Caching에는 model 이름이 'models/' 접두사 필요
# (참고: 일부 preview 모델은 caching 미지원일 수 있음 — 그때는 안정 모델 사용)

CACHE_MODEL = "gemini-3-flash-preview"

try:
    cache = client.caches.create(
        model=CACHE_MODEL,
        config=types.CreateCachedContentConfig(
            display_name="vds_2weeks_data",
            system_instruction=SYSTEM_INSTRUCTION,
            contents=[f"{METADATA}\n\n## VDS 데이터 (2주치)\n\n{csv_text_14d}"],
            ttl="3600s",  # 1시간
        ),
    )
    print(f"✅ Cache created")
    print(f"   Name:  {cache.name}")
    print(f"   Tokens: {cache.usage_metadata.total_token_count:,}")
    print(f"   Expires: {cache.expire_time}")
except Exception as e:
    print(f"❌ Cache 생성 실패: {e}")
    print("\n   Preview 모델이 caching을 지원하지 않을 수 있습니다.")
    print("   대안: 일반 호출로 진행 (캐싱 부분만 스킵)")
    cache = None

### 캐시를 사용한 반복 질의

In [ ]:
if cache:
    QUESTIONS = [
        "각 검지기의 1주차와 2주차 평균 교통량을 비교해주세요.",
        "출근 피크(7~9시) 평균 통행량 TOP 3 검지기를 알려주세요.",
        "주말 대비 평일 교통량이 가장 큰 차이를 보이는 검지기는?",
    ]

    for q in QUESTIONS:
        t0 = time.time()
        resp = client.models.generate_content(
            model=CACHE_MODEL,
            contents=q,
            config=types.GenerateContentConfig(
                cached_content=cache.name,
                thinking_config=types.ThinkingConfig(thinking_level="low"),
            ),
        )
        u = resp.usage_metadata
        cached_tokens = getattr(u, "cached_content_token_count", 0) or 0
        elapsed = time.time() - t0
        print(f"\n{'='*60}")
        print(f"Q: {q}")
        print(f"⏱  {elapsed:.2f}s | cached: {cached_tokens:,} | new in: {u.prompt_token_count - cached_tokens:,} | out: {u.candidates_token_count}")
        print(f"{'='*60}")
        print(resp.text[:500])
        time.sleep(1.5)
else:
    print("(캐시 미생성으로 이 셀 스킵)")

## 3-7. 비용 비교 — 캐싱 전 vs 후

같은 컨텍스트(2주치)를 N번 질의할 때의 비용을 계산해봅니다.

In [ ]:
# 가격 (per 1M tokens)
PRICE_INPUT = 0.50      # gemini-3-flash-preview input
PRICE_OUTPUT = 3.00     # output
PRICE_CACHED = 0.125    # cached input (75% 할인 가정 — Vertex 기준은 90%)
PRICE_STORAGE_PER_HOUR = 1.00  # 캐시 저장 비용 (Flash 기준)

context_tokens = token_14d
output_tokens_per_query = 500  # 평균
cache_storage_hours = 1

print(f"컨텍스트: {context_tokens:,} 토큰\n")

print(f"{'Queries':>10} | {'Without Cache':>16} | {'With Cache':>14} | {'Savings':>10}")
print("-" * 60)
for n in [1, 5, 10, 50, 100]:
    cost_no_cache = (
        n * context_tokens * PRICE_INPUT / 1_000_000
        + n * output_tokens_per_query * PRICE_OUTPUT / 1_000_000
    )
    cost_with_cache = (
        context_tokens * PRICE_INPUT / 1_000_000  # 1회 캐시 생성 비용
        + n * context_tokens * PRICE_CACHED / 1_000_000
        + n * output_tokens_per_query * PRICE_OUTPUT / 1_000_000
        + cache_storage_hours * context_tokens * PRICE_STORAGE_PER_HOUR / 1_000_000
    )
    savings = (1 - cost_with_cache / cost_no_cache) * 100
    print(f"{n:>10} | ${cost_no_cache:>14.4f} | ${cost_with_cache:>12.4f} | {savings:>8.1f}%")

> **결론**: 같은 컨텍스트로 **3~4번 이상 질의하면 캐싱이 이득**입니다. ITS 운영 워크로드(예: "같은 매뉴얼/규정집을 기준으로 매일 다른 민원 수십 건에 답변")에서 캐싱은 거의 필수입니다.

## 3-8. 캐시 관리 — TTL 연장 / 삭제

In [ ]:
if cache:
    # TTL 연장
    updated = client.caches.update(
        name=cache.name,
        config=types.UpdateCachedContentConfig(ttl="7200s"),  # 2시간으로 연장
    )
    print(f"✅ TTL 연장: {updated.expire_time}")

    # 캐시 삭제 (강의 종료 후 비용 누적 방지)
    client.caches.delete(name=cache.name)
    print(f"✅ Cache deleted")
else:
    print("(캐시 미생성)")

## 3-9. 도전 과제

### 과제 A — 실시간 알람 시뮬레이션
VDS 데이터에서 다음 조건을 자연어 질의 한 번으로 잡아내는 프롬프트를 작성하세요:
- 평상시 대비 속도가 50% 이상 급락한 시점
- 그 직전 30분 동안의 교통량 추이
- 사고로 추정되는지 자연 정체로 추정되는지 판단 근거

### 과제 B — Long Context vs Chunking 비교
같은 질의를 두 가지 방식으로 처리하고 응답 품질을 비교하세요:
1. **Long context**: 전체 데이터 통째로 던지기 (방금 한 방식)
2. **Chunking**: 검지기별로 나눠서 5번 호출 후 결과 합치기

어느 쪽이 응답 품질이 더 좋은가? 비용은 어떻게 다른가?

## 3-10. 정리

- ✅ 1만 행 시계열 데이터를 RAG 없이 long context로 처리
- ✅ 자연어로 이상 패턴 검출 + 운영 보고서 자동 생성
- ✅ Explicit Caching으로 반복 질의 비용 최적화
- ✅ 캐시 라이프사이클 관리 (TTL, 삭제)

**핵심 교훈**: ITS 영역의 많은 분석 워크로드는 "같은 데이터/매뉴얼에 다른 질문 여러 번"입니다. 이 경우 캐싱을 안 쓰는 건 그냥 비용 누수입니다.

다음 핸즈온(`04_function_calling_agent.ipynb`)에서는 함수 호출로 외부 시스템과 연동되는 ITS 미니 에이전트를 만듭니다.